# Building Agentic AI Systems with BeeAI

This notebook follows `Instructions.md` in this folder, adapted for a local Jupyter workflow.

Changes from the original lab:

- Uses OpenAI models through BeeAI's `ChatModel` abstraction.
- Uses top-level `await`, which is the cleanest way to run BeeAI coroutines in notebooks.
- Uses a Tavily-backed custom BeeAI tool for web search when search is required.
- Keeps the examples compact so the notebook can be run end-to-end.

## 1. Environment Setup

Set `OPENAI_API_KEY` before running model-backed cells. Set `TAVILY_API_KEY` to enable live web search in the Tavily examples. The notebook loads a local `.env` file if present.

In [1]:
import json
import logging
import math
import os
import re
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field

# RequirementAgent is the main BeeAI agent class used in these examples.
from beeai_framework.agents.requirement import RequirementAgent

# ConditionalRequirement constrains tool usage: order, frequency, and forced steps.
from beeai_framework.agents.requirement.requirements.conditional import ConditionalRequirement

# ChatModel is BeeAI's provider-agnostic LLM wrapper. Message classes represent chat turns.
from beeai_framework.backend import ChatModel, ChatModelParameters, SystemMessage, UserMessage

# RunContext, Emitter, ToolRunOptions, and output classes are used when defining custom tools.
from beeai_framework.context import RunContext
from beeai_framework.emitter import Emitter

# UnconstrainedMemory keeps the agent conversation/history available during a run.
from beeai_framework.memory import UnconstrainedMemory

# GlobalTrajectoryMiddleware records tool/agent execution flow for observability.
from beeai_framework.middleware.trajectory import GlobalTrajectoryMiddleware

# Tool is the base class for custom tools. StringToolOutput is a simple text result wrapper.
from beeai_framework.tools import StringToolOutput, Tool, ToolRunOptions

# HandoffTool lets one agent delegate work to another agent.
from beeai_framework.tools.handoff import HandoffTool

# ThinkTool gives the agent an explicit reasoning/planning action.
from beeai_framework.tools.think import ThinkTool

# OpenMeteoTool is a built-in weather tool used by the travel examples.
from beeai_framework.tools.weather import OpenMeteoTool

load_dotenv()
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

MODEL_NAME = os.getenv("OPENAI_MODEL", "openai:gpt-4o-mini")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

# One shared LLM instance is reused by all examples. temperature=0 keeps outputs steadier.
llm = ChatModel.from_name(MODEL_NAME, ChatModelParameters(temperature=0))

# Small display helper so each cell prints a readable section header.
def show(title: str, text: Any) -> None:
    print(f"\n{'=' * 80}\n{title}\n{'=' * 80}")
    print(text)

# BeeAI response shapes vary slightly by component/version; this extracts the final text.
def agent_text(response: Any) -> str:
    if hasattr(response, "last_message") and response.last_message is not None:
        return response.last_message.text
    if hasattr(response, "answer") and response.answer is not None:
        return response.answer.text
    return str(response)

show("Model", MODEL_NAME)


Model
openai:gpt-4o-mini


## 2. First AI Conversation

This corresponds to the lab's first chat-model example. `ChatModel` provides a provider-agnostic interface; here it is configured for OpenAI.

In [2]:
async def basic_chat_example() -> None:
    # SystemMessage sets the assistant behavior; UserMessage is the actual request.
    messages = [
        SystemMessage("You are a helpful AI assistant and creative writing expert."),
        UserMessage(
            "Help me brainstorm a unique business idea for a food delivery service "
            "that does not exist yet."
        ),
    ]
    # llm.run(...) is asynchronous because model calls are network operations.
    response = await llm.run(messages)
    show("Basic chat", response.get_text_content())

await basic_chat_example()


Basic chat
Sure! Here are some unique food delivery service ideas that could stand out in the market:

1. **Farm-to-Table Meal Kits**: A subscription service that partners with local farms to deliver fresh, seasonal ingredients along with easy-to-follow recipes. Customers can choose from various dietary preferences (vegan, gluten-free, etc.), and the service could include a rotating selection of local wines or craft beers that pair with the meals.

2. **Cultural Cuisine Experience Boxes**: A monthly subscription box that delivers a curated selection of ingredients and recipes from a different country each month. Each box could include traditional snacks, spices, and cooking tools, along with a virtual cooking class led by a chef from that culture.

3. **Zero-Waste Meal Delivery**: A service that focuses on sustainability by delivering meals in reusable containers. Customers can return the containers for cleaning and reuse, and the menu would feature dishes made from surplus ingredient

## 3. Prompt Templates

The lab uses a simple prompt-template class. This keeps templating explicit and dependency-free.

In [3]:
class SimplePromptTemplate:
    # This tiny class mimics common prompt-template libraries without extra dependencies.
    def __init__(self, template: str):
        self.template = template

    def render(self, variables: dict[str, Any]) -> str:
        rendered = self.template
        for key in variables:
            rendered = rendered.replace(f"{{{{{key}}}}}", f"{{{key}}}")
        return rendered.format(**variables)

async def prompt_template_example() -> None:
    template = SimplePromptTemplate(
        """
You are a senior data scientist evaluating a machine learning project proposal.

Project Details:
- Project Name: {{project_name}}
- Business Problem: {{business_problem}}
- Available Data: {{data_description}}
- Timeline: {{timeline}}
- Success Metrics: {{success_metrics}}

Please provide:
1. Feasibility assessment (1-10 scale)
2. Key technical challenges
3. Recommended approach
4. Risk mitigation strategies
5. Expected outcomes
        """.strip()
    )
    scenario = {
        "project_name": "Smart Inventory Optimization",
        "business_problem": "Reduce inventory costs while maintaining 95% product availability",
        "data_description": "2 years of sales data, supplier lead times, seasonal patterns, 500K records",
        "timeline": "3 months development, 1 month testing",
        "success_metrics": "15% cost reduction, maintain 95% availability, less than 2% forecast error",
    }
    prompt = template.render(scenario)
    response = await llm.run([UserMessage(prompt)])
    show("Rendered prompt", prompt)
    show("Project evaluation", response.get_text_content())

await prompt_template_example()


Rendered prompt
You are a senior data scientist evaluating a machine learning project proposal.

Project Details:
- Project Name: Smart Inventory Optimization
- Business Problem: Reduce inventory costs while maintaining 95% product availability
- Available Data: 2 years of sales data, supplier lead times, seasonal patterns, 500K records
- Timeline: 3 months development, 1 month testing
- Success Metrics: 15% cost reduction, maintain 95% availability, less than 2% forecast error

Please provide:
1. Feasibility assessment (1-10 scale)
2. Key technical challenges
3. Recommended approach
4. Risk mitigation strategies
5. Expected outcomes

Project evaluation
### 1. Feasibility Assessment (1-10 scale)
**Score: 7/10**

The project is feasible given the available data and the defined business problem. However, the complexity of accurately forecasting demand and optimizing inventory levels while maintaining high availability poses significant challenges. The timeline of 4 months is tight, espe

## 4. Structured Output

BeeAI can request model output that conforms to a Pydantic schema.

In [4]:
class BusinessPlan(BaseModel):
    # Pydantic describes the exact structured object we want the model to return.
    business_name: str = Field(description="A concise business name.")
    target_market: str = Field(description="The primary customer segment.")
    value_proposition: str = Field(description="The main customer benefit.")
    revenue_model: str = Field(description="How the business makes money.")
    first_90_days: list[str] = Field(description="Practical launch actions.")

async def structured_output_example() -> None:
    # response_format asks BeeAI/model backend to return data matching BusinessPlan.
    response = await llm.run(
        [UserMessage("Create a concise business plan for a zero-waste meal prep service.")],
        response_format=BusinessPlan,
    )
    show("Structured business plan", json.dumps(response.output_structured.model_dump(), indent=2))

await structured_output_example()


Structured business plan
{
  "business_name": "EcoPrep Meals",
  "target_market": "Health-conscious individuals and families seeking sustainable meal options.",
  "value_proposition": "Delicious, nutritious meals prepared with zero waste, using locally sourced ingredients and eco-friendly packaging.",
  "revenue_model": "Subscription-based meal plans with options for one-time purchases and add-ons for snacks and beverages.",
  "first_90_days": [
    "Conduct market research to identify target demographics and preferences.",
    "Develop partnerships with local farms and suppliers for fresh, organic ingredients.",
    "Create a menu of seasonal meals that minimize waste and maximize flavor.",
    "Design eco-friendly packaging solutions that are compostable or reusable.",
    "Build a user-friendly website for subscriptions and meal selections.",
    "Launch a marketing campaign focusing on sustainability and health benefits.",
    "Host a soft launch with a limited number of customers

## 5. Minimal RequirementAgent

`RequirementAgent` is BeeAI's controllable agent type. This first version uses no external tools.

In [5]:
ANALYSIS_QUERY = "Analyze the cybersecurity risks of adopting generative AI in a small financial services company."

async def minimal_agent_example() -> None:
    agent = RequirementAgent(
        # llm is the model used for planning and final answers.
        llm=llm,
        # memory stores conversation state for this agent run/session.
        memory=UnconstrainedMemory(),
        instructions=(
            "You are a careful cybersecurity analyst. Provide concise, practical, "
            "risk-aware recommendations."
        ),
    )
    # agent.run(...) executes the agent loop and returns the final agent response.
    response = await agent.run(ANALYSIS_QUERY)
    show("Minimal agent", agent_text(response))

await minimal_agent_example()


Minimal agent
Adopting generative AI in a small financial services company presents several cybersecurity risks that need careful consideration:

1. **Data Privacy Risks**: Generative AI models often require large datasets for training, which may include sensitive customer information. If not properly managed, this data can be exposed to unauthorized access or breaches, violating regulations like GDPR or CCPA.

2. **Model Inversion Attacks**: Attackers may exploit generative AI models to infer sensitive information about the training data. This can lead to the exposure of confidential client data, which is particularly critical in the financial sector.

3. **Adversarial Attacks**: Generative AI can be susceptible to adversarial attacks where malicious inputs are crafted to deceive the model, potentially leading to incorrect outputs that could harm decision-making processes in financial services.

4. **Compliance and Regulatory Challenges**: Financial services are heavily regulated. Th

## 6. Tavily Search Tool

The original lab uses Wikipedia for research examples. This notebook uses Tavily for search-backed examples, because it is already in this repository's dependencies and is better suited for current web search. If `TAVILY_API_KEY` is missing, the tool returns a clear message instead of failing the whole notebook.

In [6]:
class TavilySearchInput(BaseModel):
    # Tool input schemas tell the LLM which arguments it can pass to the tool.
    query: str = Field(description="Search query to send to Tavily.")
    max_results: int = Field(default=3, ge=1, le=5, description="Maximum number of results.")

class TavilySearchTool(Tool[TavilySearchInput, ToolRunOptions, StringToolOutput]):
    # A BeeAI custom tool wraps normal Python code so an agent can call it.
    name = "TavilySearch"
    description = "Search the web for current information using Tavily."
    input_schema = TavilySearchInput

    def _create_emitter(self) -> Emitter:
        # Emitters label tool events, which helps middleware/debugging identify this tool.
        return Emitter.root().child(namespace=["tool", "search", "tavily"], creator=self)

    async def _run(
        self,
        input: TavilySearchInput,
        options: ToolRunOptions | None,
        context: RunContext,
    ) -> StringToolOutput:
        # _run is the actual tool implementation BeeAI executes after the agent selects it.
        api_key = os.getenv("TAVILY_API_KEY")
        if not api_key:
            return StringToolOutput(
                result="TAVILY_API_KEY is not set, so live web search was skipped."
            )
        from tavily import TavilyClient

        client = TavilyClient(api_key=api_key)
        raw = client.search(query=input.query, max_results=input.max_results)
        results = raw.get("results", [])
        lines = []
        for item in results:
            title = item.get("title", "Untitled")
            url = item.get("url", "")
            content = re.sub(r"\s+", " ", item.get("content", "")).strip()
            lines.append(f"- {title}: {content}\n  Source: {url}")
        return StringToolOutput(result="\n".join(lines) or "No Tavily results found.")

async def tavily_agent_example() -> None:
    agent = RequirementAgent(
        llm=llm,
        memory=UnconstrainedMemory(),
        # tools are the actions the agent may choose during its reasoning loop.
        tools=[ThinkTool(), TavilySearchTool()],
        instructions=(
            "You are a research assistant. Think briefly, search when current facts "
            "are needed, and include source URLs when search results are available."
        ),
        requirements=[
            # Force the agent to plan first, then limit search calls to control cost/noise.
            ConditionalRequirement(ThinkTool, force_at_step=1, consecutive_allowed=False),
            ConditionalRequirement(TavilySearchTool, max_invocations=2),
        ],
        # Middleware records tool calls so you can inspect the trajectory while debugging.
        middlewares=[GlobalTrajectoryMiddleware(included=[Tool])],
    )
    response = await agent.run("What are two current risks companies should consider before deploying AI agents?")
    show("Tavily-enhanced agent", agent_text(response))

await tavily_agent_example()

--> 🛠️ ThinkTool[think][start]: {"input": {"input": {"thoughts": "To identify current risks associated with deploying AI agents, I should consider various factors such as ethical concerns, data privacy, regulatory compliance, and potential operational impacts. I will focus on two significant risks: 1) Ethical and bias issues in AI decision-making, and 2) Data privacy and security concerns. I will look for recent information to support these points."}}}
<-- 🛠️ ThinkTool[think][success]: "OK"
--> 🛠️ TavilySearchTool[TavilySearch][start]: {"input": {"input": {"query": "current risks companies should consider before deploying AI agents 2026", "max_results": 5}}}
<-- 🛠️ TavilySearchTool[TavilySearch][success]: "- State of AI trust in 2026: Shifting to the agentic era - McKinsey: Security and risk concerns are the top barrier to scaling agentic AI. Inaccuracy and cybersecurity remain the most frequently cited AI risks as\n  Source: https://www.mckinsey.com/capabilities/tech-and-ai/our-insigh

## 7. Controlled Execution and ReAct-Style Reasoning

Requirements control when and how often tools can be used. Here `ThinkTool` is forced first and after tool use, which creates a ReAct-like think -> act -> observe pattern.

In [7]:
async def controlled_reasoning_example() -> None:
    agent = RequirementAgent(
        llm=llm,
        memory=UnconstrainedMemory(),
        tools=[ThinkTool(), TavilySearchTool()],
        instructions=(
            "Analyze the problem systematically. Use search only if external facts "
            "would materially improve the answer."
        ),
        requirements=[
            # This creates a ReAct-like pattern: think first, then think again after tool use.
            ConditionalRequirement(
                ThinkTool,
                force_at_step=1,
                force_after=Tool,
                consecutive_allowed=False,
                max_invocations=3,
            ),
            # Search is only allowed after thinking and only once in this compact example.
            ConditionalRequirement(TavilySearchTool, only_after=[ThinkTool], max_invocations=1),
        ],
        middlewares=[GlobalTrajectoryMiddleware(included=[Tool])],
    )
    response = await agent.run(
        "Create a short risk checklist for using AI agents in customer support operations."
    )
    show("Controlled ReAct-style agent", agent_text(response))

await controlled_reasoning_example()

--> 🛠️ ThinkTool[think][start]: {"input": {"input": {"thoughts": "To create a risk checklist for using AI agents in customer support operations, I need to consider various aspects such as data privacy, accuracy, user experience, and compliance. The checklist should be concise and cover the most critical risks associated with AI in this context. I will outline potential risks and categorize them for clarity."}}}
<-- 🛠️ ThinkTool[think][success]: "OK"
--> 🛠️ FinalAnswerTool[final_answer][start]: {"input": {"input": {"response": "### Risk Checklist for Using AI Agents in Customer Support Operations\n\n1. **Data Privacy and Security**  \n   - Ensure compliance with data protection regulations (e.g., GDPR, CCPA).  \n   - Implement strong data encryption and access controls.  \n   - Regularly audit data handling practices to prevent breaches.\n\n2. **Accuracy and Reliability**  \n   - Monitor AI responses for accuracy and relevance.  \n   - Establish a feedback loop for continuous improvemen

## 8. Custom Calculator Tool

This corresponds to the lab's custom-tool section. A BeeAI tool has a Pydantic input schema, a `Tool` subclass, and an async `_run` method.

In [8]:
class CalculatorInput(BaseModel):
    # The calculator accepts one string expression from the agent.
    expression: str = Field(description="A simple arithmetic expression, such as '12 * (4 + 5)'.")

class SimpleCalculatorTool(Tool[CalculatorInput, ToolRunOptions, StringToolOutput]):
    # Tool[...] declares: input schema, run options type, and output type.
    name = "SimpleCalculator"
    description = "Safely evaluates simple arithmetic expressions."
    input_schema = CalculatorInput

    def _create_emitter(self) -> Emitter:
        return Emitter.root().child(namespace=["tool", "math", "calculator"], creator=self)

    def _safe_calculate(self, expression: str) -> float:
        # Keep eval tightly sandboxed and limited to simple arithmetic/math helpers.
        allowed = {"abs": abs, "round": round, "sqrt": math.sqrt, "pow": pow}
        if not re.fullmatch(r"[0-9+\-*/()., sqrtpowabslround\s]+", expression):
            raise ValueError("Expression contains unsupported characters.")
        return float(eval(expression, {"__builtins__": {}}, allowed))

    async def _run(
        self,
        input: CalculatorInput,
        options: ToolRunOptions | None,
        context: RunContext,
    ) -> StringToolOutput:
        # Returning StringToolOutput gives the agent a text observation to reason over.
        try:
            result = self._safe_calculate(input.expression)
            return StringToolOutput(result=f"{input.expression} = {result:g}")
        except Exception as exc:
            return StringToolOutput(result=f"Calculation error: {exc}")

async def calculator_agent_example() -> None:
    agent = RequirementAgent(
        llm=llm,
        memory=UnconstrainedMemory(),
        # The agent can decide to call this calculator instead of doing arithmetic in text.
        tools=[SimpleCalculatorTool()],
        instructions="Use the calculator tool for arithmetic. Explain the result briefly.",
    )
    response = await agent.run("A subscription costs 29.99 per month. What is the annual cost plus 8 percent tax?")
    show("Calculator agent", agent_text(response))

await calculator_agent_example()


Calculator agent
The annual cost of the subscription, including an 8% tax, is $388.67.


## 9. Multi-Agent Travel Planner

This mirrors the lab's final multi-agent example: specialized agents are connected through `HandoffTool`, and a coordinator synthesizes their work. Search is handled by the Tavily tool.

In [10]:
async def multi_agent_travel_planner() -> None:
    # Specialist 1: researches destination facts and practical travel context.
    destination_expert = RequirementAgent(
        llm=llm,
        tools=[ThinkTool(), TavilySearchTool()],
        memory=UnconstrainedMemory(),
        instructions=(
            "You are a destination research expert. Focus on attractions, practical travel context, "
            "and safety considerations. Use search for current or factual details."
        ),
        requirements=[
            ConditionalRequirement(ThinkTool, force_at_step=1, consecutive_allowed=False),
            ConditionalRequirement(TavilySearchTool, max_invocations=2),
        ],
    )

    # Specialist 2: handles weather and packing/activity implications.
    travel_meteorologist = RequirementAgent(
        llm=llm,
        tools=[ThinkTool(), OpenMeteoTool()],
        memory=UnconstrainedMemory(),
        instructions=(
            "You are a travel meteorologist. Provide weather-aware packing and activity guidance."
        ),
        requirements=[
            ConditionalRequirement(ThinkTool, force_at_step=1, consecutive_allowed=False),
            ConditionalRequirement(OpenMeteoTool, max_invocations=1),
        ],
    )

    # Specialist 3: handles language, etiquette, and cultural guidance.
    language_expert = RequirementAgent(
        llm=llm,
        tools=[ThinkTool(), TavilySearchTool()],
        memory=UnconstrainedMemory(),
        instructions=(
            "You are a language and cultural etiquette expert. Give practical phrases, customs, "
            "and respectful communication advice."
        ),
        requirements=[ConditionalRequirement(ThinkTool, force_at_step=1, consecutive_allowed=False)],
    )

    # The coordinator is the user-facing agent. It delegates to specialists via HandoffTool.
    coordinator = RequirementAgent(
        llm=llm,
        memory=UnconstrainedMemory(),
        tools=[
            ThinkTool(),
            # Each HandoffTool exposes a specialist agent as a callable tool.
            HandoffTool(
                target=destination_expert,
                name="DestinationResearch",
                description="Consult for destination attractions, logistics, safety, and practical guidance.",
            ),
            HandoffTool(
                target=travel_meteorologist,
                name="WeatherPlanning",
                description="Consult for weather, packing, and activity planning.",
            ),
            HandoffTool(
                target=language_expert,
                name="LanguageCulturalGuidance",
                description="Consult for phrases, etiquette, customs, and cultural awareness.",
            ),
        ],
        instructions=(
            "You are the travel coordinator. Delegate to specialists when helpful, then synthesize "
            "a concise travel plan with destination, weather, and cultural guidance."
        ),
        requirements=[ConditionalRequirement(ThinkTool, force_at_step=1, consecutive_allowed=False)],
        middlewares=[GlobalTrajectoryMiddleware(included=[Tool])],
    )

    query = (
        "I am planning a 10-day first-time cultural trip to Japan, split between Tokyo and Kyoto. "
        "I speak English only. What destination, weather, and etiquette guidance should I know?"
    )
    response = await coordinator.run(query)
    show("Multi-agent travel planner", agent_text(response))

await multi_agent_travel_planner()

--> 🛠️ ThinkTool[think][start]: {"input": {"input": {"thoughts": "To create a comprehensive travel plan for a 10-day trip to Japan, I need to gather information on the following aspects: 1. Destination highlights in Tokyo and Kyoto, including cultural sites, activities, and experiences. 2. Weather conditions during the travel period to suggest appropriate clothing and activities. 3. Cultural etiquette and language tips for an English speaker to navigate social interactions in Japan. I will need to consult the relevant tools for each of these areas."}}}
<-- 🛠️ ThinkTool[think][success]: "OK"
--> 🛠️ HandoffTool[destinationresearch][start]: {"input": {"input": {"task": "Provide cultural attractions and activities in Tokyo and Kyoto for a first-time visitor."}}}
    --> 🛠️ ThinkTool[think][start]: {"input": {"input": {"thoughts": "I need to identify key cultural attractions and activities in both Tokyo and Kyoto that would be suitable for a first-time visitor. This includes historical site

## What This Notebook Covered

- Chat model usage with OpenAI via BeeAI `ChatModel`.
- Prompt templates.
- Structured output with Pydantic.
- Minimal `RequirementAgent` usage.
- Tool-enabled agents with Tavily search.
- Controlled execution using `ConditionalRequirement` and `ThinkTool`.
- Custom BeeAI tools.
- Multi-agent coordination with `HandoffTool`.